# Inspect a saved S/N map — PIRATE 1.5

       This applies the retained full-band peak finder to one tree of a version-3
       ASDF map. The embedded config, plan, Dcores and token encoding are authoritative.
       Select the `pirate-15-validation` environment as the notebook kernel.
       Set `ASDF_PATH` below to a freshly generated map. Older maps require the
       preserved 1.4 environment or regeneration from raw acquisition frames.

In [ ]:
from pathlib import Path
import os
import sys
import pirate_frb

# Select the validation environment as the notebook kernel before running.
REPO_ROOT = Path(os.environ.get('PIRATE_REPO',
    Path(pirate_frb.__file__).resolve().parents[1])).expanduser().resolve()
if Path(pirate_frb.__file__).resolve().parents[1] != REPO_ROOT:
    raise RuntimeError('The kernel imported another PIRATE checkout. Select the 1.5 environment and restart the kernel.')
sys.path.insert(0, str(REPO_ROOT))
print('PIRATE:', pirate_frb.__file__)

In [ ]:
GPU_DEVICE = 0
TREE_INDEX = 0
SNR_THRESHOLD = 10.0
DM_REACH = 8
WAIST_BINS = 1
EDGE_POLICY = 'exclude'
DM_REACH_VALUES = (1, 2, 4, 8, 16)
MAX_TABLE_ROWS = 50
TIME_WINDOW_S = None
DM_WINDOW = None
TIMING_WARMUP_RUNS = 3
TIMING_MEASURED_RUNS = 20

ASDF_PATH = Path(os.environ.get('PIRATE_SNR_MAP',
    REPO_ROOT / 'validation/acq/frame_b100_t2_snrmap.asdf')).expanduser()

In [ ]:
import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
import peakfinder_tests.peakfinders as pf

if pf.METHODS != ('full_band_bowtie',):
    raise RuntimeError('This notebook expects the retained production full-band peak finder.')
cp.cuda.Device(GPU_DEVICE).use()

import asdf
import operator
from pirate_frb.ArgmaxMetadata import read_argmax_metadata
from pirate_frb.run_offline_dedisperser import _validate_snr_asdf_tree

with asdf.open(ASDF_PATH, lazy_load=True) as af:
    root = af.tree
    plan = _validate_snr_asdf_tree(root)
    dcores = read_argmax_metadata(
        root, ntrees=int(plan.ntrees),
        douts=[int(t.nt_ds) // int(t.nt_out) for t in plan.trees])
    if not 0 <= TREE_INDEX < int(plan.ntrees):
        raise ValueError('TREE_INDEX is outside the saved plan.')
    source = dict(root['source'])
    if isinstance(source['time_chunk_index'], (bool, np.bool_)):
        raise ValueError('Source chunk must be an integer.')
    time_chunk_index = operator.index(source['time_chunk_index'])
    if time_chunk_index < 0:
        raise ValueError('Source chunk must be nonnegative.')
    producer_start = root.get('producer_start_time_chunk_index')
    snr_cpu = np.array(root['trees'][TREE_INDEX]['snr'], copy=True)
    argmax_cpu = np.array(root['trees'][TREE_INDEX]['argmax'], copy=True)
    tree_rows = [dict(tree=i, shape=(int(t.ndm_out), int(t.nt_out)),
                      dm_min=float(t.dm_min), dm_max=float(t.dm_max), dcore=dcores[i])
                 for i, t in enumerate(plan.trees)]
time_sample_s, nt_in = float(plan.config.time_sample_ms) / 1000, int(plan.nt_in)
reference_freq_mhz = float(plan.config.zone_freq_edges[0])
snr_gpu, argmax_gpu = cp.asarray(snr_cpu), cp.asarray(argmax_cpu)
display(tree_rows)
print('Source:', source, 'Producer start chunk:', producer_start)
if producer_start is None:
    print('Startup provenance is unknown; this view does not establish acquisition completeness.')

## Full-band bowtie geometry

The tree sets the coarse pixel sizes:
$\Delta DM=(DM_{max}-DM_{min})/n_{DM}$ and
$\Delta t=t_{sample}\,n_{t,in}/n_{t,out}$.
For a competitor at $\delta DM$, residual dispersion gives
$\delta t(f)=4148.808\,\delta DM\,(f_{ref}^{-2}-f^{-2})$ seconds, with frequencies in MHz.
The two observing-band edges bound the bowtie. `DM_REACH` is a radius in coarse DM
bins and `WAIST_BINS` adds a discrete allowance along its ridge.

The mask below is the actual production footprint. Candidate coordinates are
decoded with the producer's Dcores and all four token bytes, including extra DM.

In [ ]:
geometry = pf.build_peakfinder_geometry(
    plan, TREE_INDEX, dcores=dcores, time_sample_s=time_sample_s,
    nt_in=nt_in, reference_freq_mhz=reference_freq_mhz,
    dm_reach=DM_REACH, waist_bins=WAIST_BINS,
)
mask = cp.asnumpy(geometry.full_band_footprint)
dm_radius, time_radius = (size // 2 for size in mask.shape)
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.imshow(mask, origin='lower', interpolation='nearest', aspect='auto',
          extent=(-time_radius - .5, time_radius + .5, -dm_radius - .5, dm_radius + .5))
ax.set(xlabel='Time offset (coarse bins)', ylabel='DM offset (coarse bins)',
       title='Production full-band bowtie')
plt.show()
print('Producer Dcores:', tuple(dcores), 'Encoding:', pf.ARGMAX_ENCODING)
print('Coarse DM step:', geometry.dm_step, 'Coarse time step (ms):', 1000 * geometry.time_step_s)

## Select and decode candidates

This is a single-map inspection. `EDGE_POLICY="exclude"` omits centres whose
footprint crosses the selected map boundary. The streaming grouper additionally
uses neighbouring chunks, startup provenance, and grouping windows; its final
event catalog can therefore differ from this list of peak-finder candidates.

In [ ]:
candidates = pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                              geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
decoded = pf.decode_candidates(
    plan, candidates, dcores=dcores, itree=TREE_INDEX,
    time_chunk_index=time_chunk_index, ntime=nt_in, time_sample_s=time_sample_s,
)
candidate_rows = [
    dict(idm=int(decoded['idm'][i]), itime=int(decoded['itime'][i]),
         snr=float(decoded['snr'][i]), dm=float(decoded['dm'][i]),
         toa_s=float(decoded['toa_ref_s'][i]), width_ms=1000 * float(decoded['width_s'][i]),
         freq_lo_MHz=float(decoded['freq_lo_MHz'][i]),
         freq_hi_MHz=float(decoded['freq_hi_MHz'][i]),
         token=f"0x{int(decoded['argmax_token'][i]):08x}")
    for i in np.argsort(-decoded['snr'])[:MAX_TABLE_ROWS]
]
print('Candidates:', len(candidates))
display(candidate_rows)

## Map and decoded candidates

       Coarse map cells provide the background grid; decoded markers carry the
       exact token-derived DM and time at the lowest observing-band edge.

In [ ]:
tree = plan.trees[TREE_INDEX]
dm_edges = float(tree.dm_min) + np.arange(snr_gpu.shape[0] + 1) * geometry.dm_step
time_edges = time_chunk_index * nt_in * time_sample_s + np.arange(snr_gpu.shape[1] + 1) * geometry.time_step_s
snr_cpu = cp.asnumpy(snr_gpu)
finite = snr_cpu[np.isfinite(snr_cpu)]
if not finite.size:
    raise ValueError('The selected map contains no finite S/N values.')
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
image = ax.pcolormesh(time_edges, dm_edges, snr_cpu, shading='flat', cmap='viridis',
                     vmin=float(np.percentile(finite, 1)), vmax=float(np.max(finite)))
fig.colorbar(image, ax=ax, label='S/N')
ax.scatter(decoded['toa_ref_s'], decoded['dm'], facecolors='none', edgecolors='red',
           label='Decoded candidate', s=65)
if 'INJECTED_TOAS_S' in globals():
    ax.scatter(INJECTED_TOAS_S, np.full(len(INJECTED_TOAS_S), INJECTED_DM),
               marker='x', color='white', label='Injected burst')
ax.set(xlabel=f'Time since sequence start (s), reference {reference_freq_mhz:g} MHz',
       ylabel='DM (pc cm$^{-3}$)', title=f'Full-band peak finder: tree {TREE_INDEX}, chunk {time_chunk_index}')
if TIME_WINDOW_S is not None:
    ax.set_xlim(*TIME_WINDOW_S)
if DM_WINDOW is not None:
    ax.set_ylim(*DM_WINDOW)
ax.legend()
plt.show()

## Effect of changing the DM reach

In [ ]:
reach_rows = []
for reach in DM_REACH_VALUES:
    reach_geometry = pf.build_peakfinder_geometry(
        plan, TREE_INDEX, dcores=dcores, time_sample_s=time_sample_s,
        nt_in=nt_in, reference_freq_mhz=reference_freq_mhz,
        dm_reach=reach, waist_bins=WAIST_BINS,
    )
    found = pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                             reach_geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
    reach_rows.append(dict(dm_reach=reach, candidates=len(found),
                           footprint_shape=tuple(reach_geometry.full_band_footprint.shape)))
display(reach_rows)

## Local timing diagnostic

       This measures peak finding only, excluding file reads, transfer, decoding
       and grouping. A single-map timing is not an end-to-end throughput result.

In [ ]:
if TIMING_WARMUP_RUNS < 0 or TIMING_MEASURED_RUNS < 1:
    raise ValueError('Timing needs nonnegative warmup and at least one measurement.')
for _ in range(TIMING_WARMUP_RUNS):
    pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                     geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
gpu_times_ms = []
for _ in range(TIMING_MEASURED_RUNS):
    start, stop = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    pf.run_peakfinder('full_band_bowtie', snr_gpu, argmax_gpu,
                     geometry, SNR_THRESHOLD, edge_policy=EDGE_POLICY)
    stop.record()
    stop.synchronize()
    gpu_times_ms.append(cp.cuda.get_elapsed_time(start, stop))
print(f'Per-map peak-finding GPU time: median {np.median(gpu_times_ms):.3f} ms, '
      f'{len(gpu_times_ms)} measurements. This excludes generation, decoding and grouping.')